# Using the library

In this tutorial, we will show how to use the library to load the data and make some customized plots. We first import the relevant functions and variables from the library and check what limits and theories are available.

In [ ]:
from eor_limits import (
    KNOWN_LIMITS,
    KNOWN_THEORIES,
    load_limit_data,
    load_theory_model,
    plot_vs_k,
)

# KNOWN_LIMITS and KNOWN_THEORIES are both dictionaries with the keys being the names
# of the limits and theories and the values being the corresponding to its file paths
# (The paths are mostly for internal use but provide transparency on data storage)

print("* Available limits:", list(KNOWN_LIMITS.keys()))
print("* Available theories:", list(KNOWN_THEORIES.keys()))

## Looking at the data

Let's import the latest HERA limits and see what they look like! The limits are stored as a `DataSet` object, which contains both the metadata — telescope, author, year and DOI — and a `Data` object with the following attributes:
- `z`: a 1D numpy array of the redshift bins.
- `k`: a tuple of 1D numpy arrays of the $k$-bins for each redshift bin.
- `delta_squared`: a tuple of 1D numpy arrays of the $\Delta_{21}^2$ limits for each redshift bin.

alongside other attributes such as `z_lower`, `z_upper`, `k_lower`, `k_upper` for the bin edges (if available). 

A useful method of the `Data` object is `as_pandas_df()`, which converts the data into a pandas DataFrame for nicer display. The rows of the pandas DataFrame are indexed by the redshift bins, and each contain the $k$-bins and $\Delta_{21}^2$ limits for that redshift bin.

In [ ]:
hera2026 = load_limit_data("HERA2026")  # or eor_limits.DataSet.load("HERA2026")
print("Telescope:", hera2026.telescope)
print("Author:", hera2026.author)
print("Year:", hera2026.year)
print("DOI:", hera2026.doi)
print("Data:")
hera2026.data.as_pandas_df()

## Slicing the data

The library allows you to do cuts on the data, which can be quite useful for comparing datasets. There are seven main functions:
- `select_closest_z(z)`: selects the redshift bin closest to the input $z$.
- `select_closest_k(k)`: selects the $k$-bin closest to the input $k$.
- `select_z_range(z_min, z_max)`: selects the redshift bins in the range $z_\mathrm{min} \leq z \leq z_\mathrm{max}$.
- `select_k_range(k_min, k_max)`: selects the $k$-bins in the range $k_\mathrm{min} \leq k \leq k_\mathrm{max}$.
- `select_delta_squared_range(delta_squared_min, delta_squared_max)`: selects the limits in the range $\Delta_{21,\mathrm{min}}^2 \leq \Delta_{21}^2 \leq \Delta_{21,\mathrm{max}}^2$.
- `select_lowest_delta_squared()`: selects the lowest limit in each redshift bin.

Let's do some cuts on the data to select only the limits in the range $z=7-10$ and $k=0.1-1$ h/Mpc.

In [ ]:
hera2026_trunc = hera2026.select_z_range(7, 10).select_k_range(0.1, 1)
hera2026_trunc.data.as_pandas_df()

What about looking at a specific $k$-bin or $z$-bin? Let's try selecting the redshift bin closest to $z=7$ and also, separately, the $k$-bin closest to $k=0.1$ h/Mpc.

In [ ]:
hera2026_z7 = hera2026.select_closest_z(7)
hera2026_z7.data.as_pandas_df()

In [ ]:
hera2026_k01 = hera2026.select_closest_k(0.1)
hera2026_k01.data.as_pandas_df()

## Custom plotting

Let's plot the limits, now including other HERA limits and some theoretical models for comparison. The theory models are loaded in the same manner as the limits so we can filter and plot them all together as shown in the next cell.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

datasets = [
    load_limit_data("HERA2022"),
    load_limit_data("HERA2023"),
    load_limit_data("HERA2026"),
    load_theory_model("Mesinger2016Faint"),
    load_theory_model("Mesinger2016Bright"),
]
labels = [
    "Phase 1 preliminary",
    "Phase 1 final",
    "Phase 2 preliminary",
    "Faint galaxies",
    "Bright galaxies",
]
colors = ["C0", "C1", "C2", "C3", "C4"]
markers = ["v", "v", "v", "none", "none"]

for dataset, label, color, marker in zip(
    datasets, labels, colors, markers, strict=True
):
    # Plotting the limits for the redshift bin closest to z=7
    # (k range included to limit the theory models to the data range)
    data_z7 = dataset.select_closest_z(7).select_k_range(1e-1, 5).data
    ax1.plot(
        data_z7.k[0],
        data_z7.delta_squared[0],
        marker=marker,
        color=color,
        label=label + f" (z={data_z7.z[0]:.2f})",
    )

    # Plotting the limits for the k-bin closest to k=0.1 h/Mpc
    # (z range included to limit the theory models to the data range)
    data_k02 = dataset.select_closest_k(0.1).select_z_range(6, 30).data
    ax2.plot(
        data_k02.z, data_k02.delta_squared, marker=marker, color=color, label=label
    )

    # Some shading for the theory models
    if "galaxies" in label:
        ax1.fill_between(data_z7.k[0], data_z7.delta_squared[0], alpha=0.1, color=color)

# Some plot formatting
ax1.set_title(r"at $z \approx 7$")
ax1.set_xlabel(r"Scale $k$ (h/Mpc)")
ax1.set_xscale("log")
ax2.set_title(r"at $k \approx 0.1$ h/Mpc")
ax2.set_xlabel(r"Redshift $z$")
ax1.legend()
ax2.legend()
for ax in [ax1, ax2]:
    ax.set_ylabel(r"$\Delta_{21}^2$ (mK$^2$)")
    ax.set_yscale("log")
    ax.grid()

fig.suptitle("HERA Power Spectrum Limits")
fig.tight_layout()
fig.show()

## In-built plotting

A similar plot can be made using the in-built plotting function `plot_vs_k()`, which takes care of all the slicing and styling for you. This plot differs slightly, in that we are not making narrow selection cuts but plotting all the data and coloured by the redshift bins.

In [ ]:
fig = plot_vs_k(
    limits=["HERA2022", "HERA2023", "HERA2026"],
    bold_limits=["HERA2026"],
    shade_limits=True,
    base_limit_style={"linewidth": 5, "shade_alpha": 0.1},
    limit_styles={"HERA2023": {"shade_alpha": 0.25, "shade_color": "C1"}},
    theories=["Mesinger2016Faint", "Mesinger2016Bright"],
    shade_theories=True,
    base_theory_style={"linestyle": "-."},
    theory_styles={
        "Mesinger2016Faint": {"color": "C3", "shade_alpha": 0.5, "shade_color": "C3"},
        "Mesinger2016Bright": {"color": "C4", "shade_alpha": 0.1, "shade_color": "C4"},
    },
    z_range=(7, 8),
    fontsize=20,
    fig_ratio=0.5,
)
fig.show()